[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ach0000/memorization_generalization_in_diffusion_models_tests/blob/main/tp/TP_entrainement_debruiteurs.ipynb)

# Entraîner soi-même : de la mémorisation à la généralisation

On entraîne de petits UNet sans biais (l'architecture des auteurs, réduite) sur des jeux de taille $N$ croissante, deux par deux sur des sous-ensembles **disjoints** A et B, puis on reproduit les figures 1 et 2 de l'article à plus petite échelle.

**Sur Colab** : `Exécution > Modifier le type d'exécution > GPU T4`, puis exécuter la cellule d'installation.

In [ ]:
# Installation (Colab) : à exécuter en premier ; en local, ne fait que rendre le code importable
GITHUB_USER = "ach0000"  # propriétaire du fork
REPO = "memorization_generalization_in_diffusion_models_tests"
BRANCH = "main"
# seuls ces fichiers du dépôt sont téléchargés (motifs de git sparse-checkout)
FICHIERS = ['/code/',
            '/tp/tp_outils.py',
            '/tp/prepare_celeba80x80.py']

import os, sys, subprocess
if "google.colab" in sys.modules:
    racine = os.path.join("/content", REPO)
    if not os.path.isdir(os.path.join(racine, ".git")):
        subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none", "--no-checkout", "-b", BRANCH,
                        "https://github.com/%s/%s.git" % (GITHUB_USER, REPO), racine], check=True)
    subprocess.run(["git", "-C", racine, "sparse-checkout", "set", "--no-cone", *FICHIERS], check=True)
    subprocess.run(["git", "-C", racine, "checkout", BRANCH], check=True)
    try:
        import pywt   # PyWavelets (générateur d'images C^alpha des auteurs) : absent de certaines images Colab
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "PyWavelets"], check=True)
else:
    racine = os.getcwd()
    while not os.path.exists(os.path.join(racine, "code", "main.py")) and os.path.dirname(racine) != racine:
        racine = os.path.dirname(racine)
os.makedirs(os.path.join(racine, "tp"), exist_ok=True)   # absent du clone partiel
os.chdir(os.path.join(racine, "tp"))
for d in ("code", "tp"):
    if os.path.join(racine, d) not in sys.path:
        sys.path.insert(0, os.path.join(racine, d))
print("Répertoire courant :", os.getcwd())

## 0. Réglages

Deux profils, choisis automatiquement :

* **gpu** (Colab) : visages CelebA 40×40, UNet à 2 blocs et 64 canaux (la configuration des auteurs pour
  40×40), $N$ jusqu'à 10 000 ;
* **cpu** (ordinateur personnel) : images 32×32, UNet à 2 blocs et 16 canaux, $N$ jusqu'à 1 000, moins
  d'itérations (environ une demi-heure en tout).

Avec trop peu d'itérations, les réseaux débruitent mal les grands bruits et l'échantillonnage diverge
(le niveau de bruit estimé augmente au lieu de diminuer) : les échantillons divergents sont écartés et comptés.

`SOURCE = "celeba"` télécharge un fichier de CelebA (environ 460 Mo) ; `SOURCE = "calpha"` n'en demande aucun.
Les modèles entraînés sont enregistrés dans `resultats/` et rechargés s'ils existent (sur Colab, ce dossier
disparaît avec la session).

In [ ]:
%matplotlib inline
import os
import numpy as np
import torch
import matplotlib.pyplot as plt
import tp_outils as T

PROFIL = "gpu" if T.DEVICE.type == "cuda" else "cpu"
SOURCE = "celeba"     # ou "calpha"
GRAINE = 0
if PROFIL == "gpu":
    K, NB_BLOCS, NB_NOYAUX, N_ITERATIONS, TAILLE_LOT = 40, 2, 64, 10000, 128
    N_LISTE = [1, 10, 100, 1000, 10000]
else:
    K, NB_BLOCS, NB_NOYAUX, N_ITERATIONS, TAILLE_LOT = 32, 2, 16, 4000, 32
    N_LISTE = [1, 10, 100, 1000]
N_MAX, N_TEST = max(N_LISTE), 300
DOSSIER = "resultats/entrainement_%s_%s_K%d_b%d_c%d_it%d" % (PROFIL, SOURCE, K, NB_BLOCS, NB_NOYAUX, N_ITERATIONS)
os.makedirs(DOSSIER, exist_ok=True)
T.nommer_figures("TP_entrainement_debruiteurs_" + PROFIL)   # figures nommées dans resultats/figures/
print("profil", PROFIL, "| calcul sur", T.DEVICE, "| modèles dans", DOSSIER)

## 1. Données : deux réservoirs disjoints et un jeu de test

Le jeu A de taille $N$ est formé des $N$ premières images du réservoir A, le jeu B des $N$ premières du
réservoir B ; les deux réservoirs n'ont aucune image en commun, et le jeu de test est distinct des deux.

In [ ]:
n_total = 2 * N_MAX + N_TEST
if SOURCE == "celeba":
    tout = T.charger_celeba(taille=K, n=n_total)
else:
    tout = T.images_calpha(4, n_total, K, graine=GRAINE)
reservoir = {"A": tout[:N_MAX], "B": tout[N_MAX:2 * N_MAX]}
X_test = tout[2 * N_MAX:]
print("réservoirs", tuple(reservoir["A"].shape), tuple(reservoir["B"].shape), "test", tuple(X_test.shape))
T.afficher_images(torch.cat([reservoir["A"][:5], reservoir["B"][:5]]), ncol=10,
                  titre="5 images du réservoir A, puis 5 du réservoir B")

## 2. Entraînement

Perte des auteurs : $\frac1B\sum_b \|r_\theta(y_b) - (y_b - x_b)\|^2$ (le réseau prédit le bruit), $\sigma$
tiré par image comme chez eux ($\sigma = u^2$, $u$ uniforme sur $[0, 1]$), Adam. Le nombre d'itérations est
le même pour tous les $N$ : les petits jeux sont parcourus un très grand nombre de fois.

In [ ]:
modeles = {}
for N in N_LISTE:
    for groupe in "AB":
        chemin = os.path.join(DOSSIER, "unet_N%d_%s.pt" % (N, groupe))
        T.fixer_graine(GRAINE)   # initialisation du réseau indépendante de ce qui précède
        modele = T.petit_unet(NB_BLOCS, NB_NOYAUX)
        if os.path.exists(chemin):
            modele.load_state_dict(torch.load(chemin, map_location=T.DEVICE))
            modele.eval()
            print("N = %5d, groupe %s : rechargé" % (N, groupe))
        else:
            print("N = %5d, groupe %s : entraînement (%d paramètres)" % (N, groupe, T.nombre_parametres(modele)))
            T.entrainer(modele, reservoir[groupe][:N], n_iterations=N_ITERATIONS, taille_lot=TAILLE_LOT,
                        graine=GRAINE, afficher_tous=N_ITERATIONS // 3)
            torch.save(modele.state_dict(), chemin)
        modeles[(N, groupe)] = modele

## 3. Figure 1 : débruitage sur l'entraînement et sur le test

In [ ]:
sig_range = np.logspace(0.3, 2.4, 10) / 255
courbes_train, courbes_test = {}, {}
for N in N_LISTE:
    f = lambda y, s, m=modeles[(N, "A")]: y - m(y)
    images_train = reservoir["A"][:N].repeat(int(np.ceil(200 / N)), 1, 1, 1)[:200]
    courbes_train["N = %d" % N] = T.courbe_psnr(f, images_train, sig_range)
    courbes_test["N = %d" % N] = T.courbe_psnr(f, X_test, sig_range)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
T.trace_psnr(courbes_train, sig_range, "Entraînement", ax=axes[0])
T.trace_psnr(courbes_test, sig_range, "Test", ax=axes[1])
T.montrer("psnr")
for N in N_LISTE:
    print("N = %5d  écart moyen entraînement - test : %5.1f dB" %
          (N, np.mean(courbes_train["N = %d" % N] - courbes_test["N = %d" % N])))

## 4. Figure 2 : deux modèles, même bruit initial

Pour chaque $N$, les modèles A et B partent de la même image initiale (moyenne des deux jeux d'entraînement
A et B réunis, plus un bruit blanc ; les auteurs partent de la moyenne du jeu de chaque modèle) et reçoivent les
mêmes tirages aléatoires pendant l'échantillonnage. Les échantillons divergents sont écartés. On compare leurs échantillons entre
eux, et chacun à l'image la plus proche de son jeu d'entraînement.

In [ ]:
n_ech = 16
resultats = {}
for N in N_LISTE:
    moyenne = torch.cat([reservoir["A"][:N], reservoir["B"][:N]]).mean(dim=0)
    init = T.init_echantillons(moyenne, n_ech, graine=GRAINE + 1)
    ech, divergents = {}, torch.zeros(n_ech, dtype=torch.bool)
    for g in "AB":
        ech[g], infos = T.echantillonner(modeles[(N, g)], init, sigma_fin=0.05, graine=GRAINE + 1)
        ech[g] = ech[g].cpu()
        divergents |= infos["divergents"]
    print("N = %5d : %d paires sur %d écartées (un des deux échantillons a divergé)" % (N, divergents.sum(), n_ech))
    ech = {g: ech[g][~divergents] for g in "AB"}
    sim_AB = T.similarites_cosinus(ech["A"], ech["B"]).diag()
    sim_train = {g: T.plus_proches_voisins(ech[g], reservoir[g][:N]) for g in "AB"}
    resultats[N] = (ech, sim_AB, sim_train)
    if len(sim_AB) == 0:
        continue
    print("N = %5d : similarité A-B médiane %.3f ; similarité au plus proche voisin d'entraînement médiane %.3f"
          % (N, sim_AB.median(), sim_train["A"][0].median()))

lignes, titres = [], []
for N in N_LISTE:
    ech, sim_AB, sim_train = resultats[N]
    if len(ech["A"]) == 0:
        continue
    lignes += [reservoir["A"][:N][sim_train["A"][1][0]], ech["A"][0], ech["B"][0], reservoir["B"][:N][sim_train["B"][1][0]]]
    titres += ["N=%d : voisin A" % N, "échantillon A", "échantillon B", "voisin B"]
T.afficher_images(torch.stack(lignes), titres, ncol=4, taille=1.8,
                  titre="Pour chaque N : plus proche image de A, échantillon A, échantillon B, plus proche image de B",
                  nom="paires")

fig, axes = plt.subplots(1, len(N_LISTE), figsize=(3.6 * len(N_LISTE), 3), sharey=True)
for ax, N in zip(axes, N_LISTE):
    ech, sim_AB, sim_train = resultats[N]
    ax.hist(sim_AB.numpy(), bins=np.linspace(0, 1, 26), alpha=0.8, label="échantillons A et B")
    ax.hist(sim_train["A"][0].numpy(), bins=np.linspace(0, 1, 26), alpha=0.6, label="échantillon A et voisin")
    ax.set_title("N = %d" % N); ax.set_xlabel("similarité cosinus")
axes[-1].legend(fontsize=8)
T.montrer("similarites")